In [16]:

# ==========================================================
# STEP 1: RECONNECT TO YOUR DATABASE
# ==========================================================

import sqlite3
import pandas as pd
# Reconnect to the database you created on Day 8

conn = sqlite3.connect('school.db')

# helper functions to run queries
def run_query(sql):
     """Runs SQL and displays results as pandas table"""
     return pd.read_sql_query(sql, conn)

# quick test to check if the table is still there
print("----STUDENTS TABLE----")
run_query("SELECT first_name, last_name, grade_level, email FROM students")

import sqlite3
import pandas as pd

# 1. Connect to database
conn = sqlite3.connect('school.db')
cursor = conn.cursor()

# 2. Build the 3 tables if they don't exist
cursor.execute('CREATE TABLE IF NOT EXISTS students (student_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, email TEXT, grade_level TEXT, enrollment_date TEXT)')
cursor.execute('CREATE TABLE IF NOT EXISTS courses (course_id INTEGER PRIMARY KEY, course_name TEXT, subject TEXT, credits INTEGER, instructor TEXT)')
cursor.execute('CREATE TABLE IF NOT EXISTS enrollments (enrollment_id INTEGER PRIMARY KEY, student_id INTEGER, course_id INTEGER, semester TEXT, final_grade REAL)')

# 3. Add the sample data
students_data = [
    (1, 'Amina', 'Okafor', 'amina@school.edu', 'Year 1', '2025-09-01'),
    (2, 'James', 'Smith', 'james@school.edu', 'Year 2', '2024-09-01'),
    (3, 'Fatima', 'Bello', 'fatima@school.edu', 'Year 1', '2025-09-01'),
    (4, 'David', 'Chen', 'david@school.edu', 'Year 3', '2023-09-01'),
    (5, 'Grace', 'Eze', 'grace@school.edu', 'Year 2', '2024-09-01'),
    (6, 'Omar', 'Hassan', 'omar@school.edu', 'Year 1', '2025-09-01'),
    (7, 'Sarah', 'Williams', 'sarah@school.edu', 'Year 3', '2023-09-01'),
    (8, 'Chidi', 'Nwosu', 'chidi@school.edu', 'Year 2', '2024-09-01'),
    (9, 'Lena', 'Mueller', 'lena@school.edu', 'Year 1', '2025-09-01'),
    (10, 'Kwame', 'Asante', 'kwame@school.edu', 'Year 3', '2023-09-01')
]

courses_data = [
    (101, 'Introduction to Python', 'Computer Science', 3, 'Dr. Adewale'),
    (102, 'Calculus I', 'Mathematics', 4, 'Prof. Schmidt'),
    (103, 'English Composition', 'English', 3, 'Dr. Osei'),
    (104, 'Data Structures', 'Computer Science', 4, 'Dr. Adewale'),
    (105, 'Statistics', 'Mathematics', 3, 'Prof. Rossi')
]

enrollments_data = [
    (1, 1, 101, 'Fall 2025', 88.5),
    (2, 1, 102, 'Fall 2025', 72.0),
    (3, 2, 104, 'Fall 2025', 91.0),
    (4, 2, 105, 'Fall 2025', 65.5),
    (5, 3, 101, 'Fall 2025', 95.0),
    (6, 3, 103, 'Fall 2025', 82.0),
    (7, 4, 104, 'Fall 2025', 78.5),
    (8, 4, 105, 'Fall 2025', 88.0),
    (9, 5, 102, 'Fall 2025', 55.0),
    (10, 5, 103, 'Fall 2025', 70.0),
    (11, 6, 101, 'Fall 2025', 42.0),
    (12, 7, 104, 'Fall 2025', 96.0),
    (13, 8, 102, 'Fall 2025', 73.5),
    (14, 9, 101, 'Fall 2025', 89.0),
    (15, 10, 105, 'Fall 2025', 91.5)
]

cursor.executemany('INSERT OR IGNORE INTO students VALUES (?, ?, ?, ?, ?, ?)', students_data)
cursor.executemany('INSERT OR IGNORE INTO courses VALUES (?, ?, ?, ?, ?)', courses_data)
cursor.executemany('INSERT OR IGNORE INTO enrollments VALUES (?, ?, ?, ?, ?)', enrollments_data)
conn.commit()

# Helper query function
def run_query(sql):
    return pd.read_sql_query(sql, conn)

print("✅ DATABASE READY & POPULATED!")
run_query("SELECT COUNT(*) AS total_students FROM students")

----STUDENTS TABLE----
✅ DATABASE READY & POPULATED!


,total_students
0,10


In [10]:
# ==========================================================
# STEP 2: UNDERSTANDING JOIN TYPES
# ==========================================================

# JOIN TYPE 1: INNER JOIN (Only matching records)
# "Show me students who exist in BOTH tables"
# This is the most common join — like an intersection

print("=== INNER JOIN (Only matches) ===")
run_query(''' 
     SELECT
     s.first_name,
     s.last_name,
     e.final_grade
  FROM students s
  INNER JOIN enrollments e ON s.student_id = e.student_id
''')

=== INNER JOIN (Only matches) ===


,first_name,last_name,final_grade
0,Amina,Okafor,88.5
1,Amina,Okafor,72.0
2,James,Smith,91.0
3,James,Smith,65.5
4,Fatima,Bello,95.0
5,Fatima,Bello,82.0
6,David,Chen,78.5
7,David,Chen,88.0
8,Grace,Eze,55.0
9,Grace,Eze,70.0


In [9]:
# JOIN TYPE 2: LEFT JOIN (All from left table + matches from right)
# "Show me ALL students, even if they have no enrollments"
# This is critical for finding students who NEVER enrolled!

print("\n=== LEFT JOIN (All students, even without enrollment) ===")
run_query('''  
    SELECT
      s.first_name,
      s.last_name,
      e.final_grade
    FROM students s
    LEFT JOIN enrollments e ON s.student_id = e.student_id
''')


=== LEFT JOIN (All students, even without enrollment) ===


,first_name,last_name,final_grade
0,Amina,Okafor,72.0
1,Amina,Okafor,88.5
2,James,Smith,65.5
3,James,Smith,91.0
4,Fatima,Bello,82.0
5,Fatima,Bello,95.0
6,David,Chen,78.5
7,David,Chen,88.0
8,Grace,Eze,55.0
9,Grace,Eze,70.0


In [8]:
# JOIN TYPE 3: RIGHT JOIN (All from right table + matches from left)
# "Show me ALL enrollments, even if student info is missing"
# Less common, but useful for data quality checks

print("\n=== RIGHT JOIN (All enrollments) ===")
# Note: SQLite doesn't support RIGHT JOIN directly,
# but you can achieve it by flipping the tables:
run_query('''
    SELECT 
        s.first_name, 
        s.last_name, 
        e.final_grade
    FROM enrollments e
    LEFT JOIN students s ON e.student_id = s.student_id
''')


=== RIGHT JOIN (All enrollments) ===


,first_name,last_name,final_grade
0,Amina,Okafor,88.5
1,Amina,Okafor,72.0
2,James,Smith,91.0
3,James,Smith,65.5
4,Fatima,Bello,95.0
5,Fatima,Bello,82.0
6,David,Chen,78.5
7,David,Chen,88.0
8,Grace,Eze,55.0
9,Grace,Eze,70.0


In [6]:
# ==========================================================
# STEP 3: THE "DEAN'S REPORT" — 3-TABLE JOIN
# ==========================================================

# We need data from ALL 3 tables:
# students → student names
# courses → course names and instructors
# enrollments → grades (this is the "bridge" table)

print("=== COMPLETE STUDENT PERFORMANCE REPORT ===")

deans_report = run_query(''' 
      SELECT 
          s.first_name,
          s.last_name,
          s.grade_level,
          c.course_name,
          c.instructor,
          e.semester,
          e.final_grade

    FROM enrollments e
    JOIN students s ON e.student_id = s.student_id
    JOIN courses c ON e.course_id = c.course_id
    ORDER BY s.last_name, c.course_name
''')

deans_report

=== COMPLETE STUDENT PERFORMANCE REPORT ===


,first_name,last_name,grade_level,course_name,instructor,semester,final_grade
0,Kwame,Asante,Year 3,Statistics,Prof. Rossi,Fall 2025,91.5
1,Fatima,Bello,Year 1,English Composition,Dr. Osei,Fall 2025,82.0
2,Fatima,Bello,Year 1,Introduction to Python,Dr. Adewale,Fall 2025,95.0
3,David,Chen,Year 3,Data Structures,Dr. Adewale,Fall 2025,78.5
4,David,Chen,Year 3,Statistics,Prof. Rossi,Fall 2025,88.0
5,Grace,Eze,Year 2,Calculus I,Prof. Schmidt,Fall 2025,55.0
6,Grace,Eze,Year 2,English Composition,Dr. Osei,Fall 2025,70.0
7,Omar,Hassan,Year 1,Introduction to Python,Dr. Adewale,Fall 2025,42.0
8,Lena,Mueller,Year 1,Introduction to Python,Dr. Adewale,Fall 2025,89.0
9,Chidi,Nwosu,Year 2,Calculus I,Prof. Schmidt,Fall 2025,73.5


In [4]:
# ==========================================================
# STEP 4: ANALYTICAL QUERIES (JOIN + GROUP BY)
# ==========================================================

# QUESTION 1: Average grade per instructor
print("=== INSTRUCTOR PERFORMANCE ===")
run_query('''  
   SELECT 
   c.instructor,
   COUNT(e.student_id) AS total_students,
   ROUND(AVG(e.final_grade), 1) AS avg_grade,
   MIN(e.final_grade) AS lowest,
   MAX(e.final_grade) AS highest
   FROM enrollments e
   JOIN courses c ON e.course_id = c.course_id
   GROUP BY c.instructor
   ORDER BY avg_grade DESC

''')

=== INSTRUCTOR PERFORMANCE ===


,instructor,total_students,avg_grade,lowest,highest
0,Dr. Adewale,7,82.9,42.0,96.0
1,Prof. Rossi,3,81.7,65.5,91.5
2,Dr. Osei,2,76.0,70.0,82.0
3,Prof. Schmidt,3,66.8,55.0,73.5


In [18]:
# ==========================================================
# SQL: LEFT JOIN (All students, with or without grades)
# ==========================================================


run_query(''' 
   SELECT 
   s.first_name,
   s.last_name,
   s.grade_level,
   e.final_grade
   FROM students s
   LEFT JOIN enrollments e ON s.student_id = e.student_id
''')

,first_name,last_name,grade_level,final_grade
0,Amina,Okafor,Year 1,72.0
1,Amina,Okafor,Year 1,88.5
2,James,Smith,Year 2,65.5
3,James,Smith,Year 2,91.0
4,Fatima,Bello,Year 1,82.0
5,Fatima,Bello,Year 1,95.0
6,David,Chen,Year 3,78.5
7,David,Chen,Year 3,88.0
8,Grace,Eze,Year 2,55.0
9,Grace,Eze,Year 2,70.0


In [19]:
# ==========================================================
# PANDAS: MERGE (Left Join)
# ==========================================================

# 1. Load the raw tables into Pandas
students_df = pd.read_sql_query("SELECT * FROM students", conn)
enrollments_df = pd.read_sql_query("SELECT * FROM enrollments", conn)


# 2. Merge them using how='left'
# on='student_id' is the matching glue (just like "ON s.student_id = e.student_id")
merged_df = pd.merge(students_df, enrollments_df, on='student_id', how='left')
# 3. Display only the columns we care about
merged_df[['first_name', 'last_name', 'grade_level', 'final_grade']]

,first_name,last_name,grade_level,final_grade
0,Amina,Okafor,Year 1,88.5
1,Amina,Okafor,Year 1,72.0
2,James,Smith,Year 2,91.0
3,James,Smith,Year 2,65.5
4,Fatima,Bello,Year 1,95.0
5,Fatima,Bello,Year 1,82.0
6,David,Chen,Year 3,78.5
7,David,Chen,Year 3,88.0
8,Grace,Eze,Year 2,55.0
9,Grace,Eze,Year 2,70.0


In [23]:
run_query(''' 
    SELECT
    s.first_name,
    c.course_name,
    e.final_grade
    FROM enrollments e
    JOIN students s ON e.student_id = s.student_id
    JOIN courses c ON e.course_id = c.course_id
    ORDER BY s.first_name
''')

,first_name,course_name,final_grade
0,Amina,Introduction to Python,88.5
1,Amina,Calculus I,72.0
2,Chidi,Calculus I,73.5
3,David,Data Structures,78.5
4,David,Statistics,88.0
5,Fatima,Introduction to Python,95.0
6,Fatima,English Composition,82.0
7,Grace,Calculus I,55.0
8,Grace,English Composition,70.0
9,James,Data Structures,91.0
